# SAM2 Segmentation Workflow

**Steps:**
1. **Configure** — set `SPK` and verify paths  
2. **Select init frame** — browse a speaker video and save the reference JPEG  
3. **Load models** — SAM2 image predictor (prompting) + video predictor (propagation)  
4. **Define regions** — edit `REGION_DEFS` with the anatomical regions to track  
5. **Prompt regions** — interactively click positive/negative points on the init frame  
6. **Test propagation** — run on a random 200-frame window; view overlay inline  
7. **Export** — write `session.json` to `sam_seg/` for use with `sam2_propagate.py`  

> **Requires** `ipympl` for interactive prompting: `pip install ipympl`

# Server Config 

1. Create server port with jupyter in tmux

```
jupyter notebook --no-browser --port=8888
```

2. In separate tmux session, create server tunnel for using GPU

```
alias gpu-tunnel="ssh -N -L 8888:localhost:8888 user@your-gpu-server.com"
```

Then: 

```
source ~/.zshrc

```

In future runs, one can run: 

```
gpu-tunnel
```

Always need two tmux sessions running each of these steps. 

In the notebook, add the url for the port as a new kernel. It should save and can be accessed using localhost in the future. 

In [ ]:
import os, json

# ── Edit this ──────────────────────────────────────────────────────────
SPK = 'spk15'
# ───────────────────────────────────────────────────────────────────────

_HERE = os.path.abspath('')
with open(os.path.join(_HERE, 'sam2_gui_config.json')) as _f:
    _cfg = json.load(_f)

DATA_DIR     = _cfg['data_dir']
VIDEO_SUBDIR = _cfg.get('video_subdir', 'video/video')
CHECKPOINT   = os.path.join(_HERE, _cfg.get('checkpoint', 'checkpoints/sam2.1_hiera_large.pt'))
MODEL_CFG    = _cfg.get('model_cfg', 'configs/sam2.1/sam2.1_hiera_l.yaml')
DEVICE_STR   = _cfg.get('device', 'cuda')

VIDEO_DIR   = os.path.join(DATA_DIR, SPK, *VIDEO_SUBDIR.split('/'))
SAM_SEG_DIR = os.path.join(DATA_DIR, SPK, 'sam_seg')
os.makedirs(SAM_SEG_DIR, exist_ok=True)

print(f'Speaker  : {SPK}')
print(f'Videos   : {VIDEO_DIR}')
print(f'Output   : {SAM_SEG_DIR}')
print(f'Device   : {DEVICE_STR}')

In [ ]:
import os, json, shutil, random, tempfile
os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'

import numpy as np
import torch
import matplotlib.pyplot as plt
import ipywidgets as widgets
import cv2
from PIL import Image
from IPython.display import display, Video

%matplotlib widget

device = torch.device(
    DEVICE_STR
    if DEVICE_STR == 'cpu' or (DEVICE_STR.startswith('cuda') and torch.cuda.is_available())
    else 'cpu'
)
print(f'Using device: {device}')
if device.type == 'cuda':
    print(f'  GPU: {torch.cuda.get_device_name(device)}')

In [ ]:
from sam2.build_sam import build_sam2, build_sam2_video_predictor
from sam2.sam2_image_predictor import SAM2ImagePredictor

print('Loading SAM2 models...')
_sam2_base    = build_sam2(MODEL_CFG, CHECKPOINT, device=device)
img_predictor = SAM2ImagePredictor(_sam2_base)

vid_predictor = build_sam2_video_predictor(MODEL_CFG, CHECKPOINT, device=device)
print('Done.')

## 1 — Select & Save Init Frame

Use the dropdown to pick a video, drag the slider to the desired frame, then click **Save as init frame**.

In [ ]:
video_files = sorted(f for f in os.listdir(VIDEO_DIR) if f.lower().endswith('.avi'))
if not video_files:
    raise FileNotFoundError(f'No .avi files found in {VIDEO_DIR}')
print(f'{len(video_files)} video(s) found.')

ann_frame_idx   = 0
INIT_FRAME_PATH = os.path.join(SAM_SEG_DIR, f'{SPK}_init_frame.jpg')

_cap    = [None]
_nf     = [0]
_out    = widgets.Output()
_status = widgets.Label('Not saved yet.')

def _load_video(vid_name):
    if _cap[0] is not None:
        _cap[0].release()
    _cap[0] = cv2.VideoCapture(os.path.join(VIDEO_DIR, vid_name))
    _nf[0]  = int(_cap[0].get(cv2.CAP_PROP_FRAME_COUNT))
    _slider.max   = _nf[0] - 1
    _slider.value = 0
    _draw_frame(0)

def _draw_frame(idx):
    cap = _cap[0]
    if cap is None:
        return
    cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
    ret, frm = cap.read()
    if not ret:
        return
    with _out:
        _out.clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(7, 5))
        ax.imshow(cv2.cvtColor(frm, cv2.COLOR_BGR2RGB), cmap='gray')
        ax.set_title(f'Frame {idx} / {_nf[0] - 1}')
        ax.axis('off')
        plt.tight_layout()
        plt.show()

def _on_save(_):
    global ann_frame_idx
    idx = _slider.value
    cap = _cap[0]
    cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
    ret, frm = cap.read()
    if not ret:
        _status.value = 'Error: could not read frame.'
        return
    img = Image.fromarray(cv2.cvtColor(frm, cv2.COLOR_BGR2RGB))
    img.save(INIT_FRAME_PATH, 'JPEG', quality=95)
    ann_frame_idx = idx
    _status.value = f'Saved frame {idx} to {INIT_FRAME_PATH}'
    print(f'ann_frame_idx = {ann_frame_idx}')

_vid_dd   = widgets.Dropdown(options=video_files, description='Video:',
                              layout=widgets.Layout(width='420px'))
_slider   = widgets.IntSlider(value=0, min=0, max=1000, step=1,
                               description='Frame:', continuous_update=False,
                               layout=widgets.Layout(width='650px'))
_save_btn = widgets.Button(description='Save as init frame', button_style='success')

_slider.observe(lambda c: _draw_frame(c['new']), names='value')
_vid_dd.observe(lambda c: _load_video(c['new']), names='value')
_save_btn.on_click(_on_save)

display(_vid_dd, _slider, widgets.HBox([_save_btn, _status]), _out)
_load_video(video_files[0])

## 2 — Load Init Frame & Define Regions

Run the cell below to load the saved JPEG into the image predictor, then edit `REGION_DEFS` to name the regions you want to track.

In [ ]:
if not os.path.exists(INIT_FRAME_PATH):
    raise FileNotFoundError(
        f'Init frame not found: {INIT_FRAME_PATH}\nRun the frame-selection cell first.'
    )

image = np.array(Image.open(INIT_FRAME_PATH).convert('RGB'))
img_predictor.set_image(image)
print(f'Image set: {image.shape}  (ann_frame_idx={ann_frame_idx})')

fig, ax = plt.subplots(figsize=(7, 5))
ax.imshow(image, cmap='gray')
ax.set_title(f'{SPK} — init frame {ann_frame_idx}')
ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# ── Edit this list to match your segmentation targets ────────────────────
REGION_DEFS = [
    {'name': 'tongue_tip',  'color': '#3cb44b'},
    {'name': 'tongue_body', 'color': '#e6194b'},
    {'name': 'upper_lip',   'color': '#4363d8'},
    {'name': 'lower_lip',   'color': '#f58231'},
    {'name': 'velum',       'color': '#911eb4'},
]
# ─────────────────────────────────────────────────────────────────────────

for i, rd in enumerate(REGION_DEFS):
    rd['obj_id'] = i

# point store: {region_name: {'points': [[x,y],...], 'labels': [1/0,...]}}
regions = {rd['name']: {'points': [], 'labels': []} for rd in REGION_DEFS}

print('Regions:')
for rd in REGION_DEFS:
    print(f"  [{rd['obj_id']}] {rd['name']}  {rd['color']}")

## 3 — Interactive Prompting

- **Left-click** — add a positive point (or use the Mode toggle)  
- **Right-click** — add a negative point  
- **Middle-click** — undo the last point  
- Select the target region in the dropdown before clicking.  
- Masks update in real time. Use **Clear Region** to reset a region.

In [ ]:
def show_mask(mask, ax, obj_id=None):
    cmap  = plt.get_cmap('tab10')
    cidx  = 0 if obj_id is None else obj_id % 10
    color = np.array([*cmap(cidx)[:3], 0.5])
    h, w  = mask.shape[-2:]
    ax.imshow(mask.reshape(h, w, 1) * color.reshape(1, 1, 4))

def show_points(coords, labels, ax, marker_size=180):
    pos = coords[labels == 1]
    neg = coords[labels == 0]
    ax.scatter(pos[:, 0], pos[:, 1], color='lime', marker='*',
               s=marker_size, edgecolors='white', linewidths=1)
    ax.scatter(neg[:, 0], neg[:, 1], color='red',  marker='*',
               s=marker_size, edgecolors='white', linewidths=1)

_fig, _ax = plt.subplots(figsize=(8, 6))
_ax.imshow(image, cmap='gray')
_ax.axis('off')
plt.tight_layout()

_region_dd = widgets.Dropdown(
    options=[(rd['name'], i) for i, rd in enumerate(REGION_DEFS)],
    description='Region:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='280px'),
)
_mode_tog = widgets.ToggleButtons(
    options=[('Positive (+)', 1), ('Negative (-)', 0)],
    value=1,
    description='Mode:',
    style={'description_width': 'initial', 'button_width': '130px'},
)
_clear_btn = widgets.Button(description='Clear Region', button_style='warning',
                             layout=widgets.Layout(width='150px'))
_status = widgets.Label('Select a region and click on the image.')

def _refresh():
    _ax.cla()
    _ax.imshow(image, cmap='gray')
    for rd in REGION_DEFS:
        r = regions[rd['name']]
        if not r['points']:
            continue
        pts = np.array(r['points'], dtype=np.float32)
        lbs = np.array(r['labels'], np.int32)
        masks, _, _ = img_predictor.predict(
            point_coords=pts,
            point_labels=lbs,
            multimask_output=False,
        )
        show_mask(masks[0], _ax, obj_id=rd['obj_id'])
        show_points(pts, lbs, _ax)
    _ax.set_title('Left=positive  |  Right=negative  |  Middle=undo')
    _ax.axis('off')
    _fig.canvas.draw_idle()

def _on_click(event):
    if event.inaxes is not _ax or event.xdata is None:
        return
    idx = _region_dd.value
    rd  = REGION_DEFS[idx]
    r   = regions[rd['name']]
    if event.button == 1:
        r['points'].append([float(event.xdata), float(event.ydata)])
        r['labels'].append(int(_mode_tog.value))
    elif event.button == 3:
        r['points'].append([float(event.xdata), float(event.ydata)])
        r['labels'].append(0)
    elif event.button == 2:
        if r['points']:
            r['points'].pop()
            r['labels'].pop()
    _status.value = f"{rd['name']}: {len(r['points'])} point(s)"
    _refresh()

def _on_clear(_):
    idx = _region_dd.value
    rd  = REGION_DEFS[idx]
    regions[rd['name']]['points'].clear()
    regions[rd['name']]['labels'].clear()
    _status.value = f"Cleared {rd['name']}."
    _refresh()

_fig.canvas.mpl_connect('button_press_event', _on_click)
_clear_btn.on_click(_on_clear)
display(widgets.HBox([_region_dd, _mode_tog]))
display(widgets.HBox([_clear_btn, _status]))
plt.show()

## 4 — Test Propagation

Randomly samples 200 consecutive frames from the speaker's videos, propagates the prompts defined above, and plays the overlay video inline.

In [ ]:
N_SAMPLE = 200

active = [rd for rd in REGION_DEFS if regions[rd['name']]['points']]
if not active:
    raise RuntimeError('No regions have points yet. Complete the prompting cell first.')

# ── pick a random video + frame window ──────────────────────────────────
test_vid  = random.choice(video_files)
test_path = os.path.join(VIDEO_DIR, test_vid)

_tc = cv2.VideoCapture(test_path)
n_total  = int(_tc.get(cv2.CAP_PROP_FRAME_COUNT))
fps_vid  = _tc.get(cv2.CAP_PROP_FPS) or 25.0
h_vid    = int(_tc.get(cv2.CAP_PROP_FRAME_HEIGHT))
w_vid    = int(_tc.get(cv2.CAP_PROP_FRAME_WIDTH))
_tc.release()

n_sample  = min(N_SAMPLE, n_total)
start_idx = random.randint(0, max(0, n_total - n_sample))
end_idx   = start_idx + n_sample

print(f'Video   : {test_vid}  ({n_total} frames @ {fps_vid:.1f} fps)')
print(f'Window  : frames {start_idx}-{end_idx - 1}  ({n_sample} frames)')

# ── extract frames to a temp directory ──────────────────────────────────
_tmp = tempfile.mkdtemp(prefix='sam2_test_')
_tc2 = cv2.VideoCapture(test_path)
_tc2.set(cv2.CAP_PROP_POS_FRAMES, start_idx)
for i in range(n_sample):
    ret, frm = _tc2.read()
    if not ret:
        break
    cv2.imwrite(os.path.join(_tmp, f'{i:05d}.jpg'), frm,
                [cv2.IMWRITE_JPEG_QUALITY, 95])
_tc2.release()
print(f'Frames extracted to {_tmp}')

# ── SAM2 video propagation ──────────────────────────────────────────────
state = vid_predictor.init_state(
    video_path=_tmp,
    offload_video_to_cpu=True,
    offload_state_to_cpu=False,
)
vid_predictor.reset_state(state)

for rd in active:
    r = regions[rd['name']]
    vid_predictor.add_new_points_or_box(
        state,
        frame_idx=0,
        obj_id=rd['obj_id'],
        points=np.array(r['points'], dtype=np.float32),
        labels=np.array(r['labels'],  np.int32),
        clear_old_points=True,
    )

print('Propagating...')
segs = {}
for fid, oids, logits in vid_predictor.propagate_in_video(state):
    segs[fid] = {oid: (logits[i] > 0).cpu().numpy() for i, oid in enumerate(oids)}
print(f'Done - {len(segs)} frames segmented.')

# ── render overlay MP4 ──────────────────────────────────────────────────
def _hex_bgr(h):
    h = h.lstrip('#')
    r, g, b = int(h[0:2], 16), int(h[2:4], 16), int(h[4:6], 16)
    return np.array([b, g, r], dtype=np.float32)

overlay_path = os.path.join(SAM_SEG_DIR, f'{SPK}_test_prop.mp4')
_writer = cv2.VideoWriter(
    overlay_path, cv2.VideoWriter_fourcc(*'mp4v'), fps_vid, (w_vid, h_vid)
)
_tc3 = cv2.VideoCapture(test_path)
_tc3.set(cv2.CAP_PROP_POS_FRAMES, start_idx)

for i in range(n_sample):
    ret, frm = _tc3.read()
    if not ret:
        break
    canvas = frm.astype(np.float32)
    if i in segs:
        for rd in active:
            mask = segs[i].get(rd['obj_id'])
            if mask is None:
                continue
            m = np.squeeze(mask).astype(bool)
            if m.shape != (h_vid, w_vid):
                continue
            canvas[m] = 0.55 * canvas[m] + 0.45 * _hex_bgr(rd['color'])
    _writer.write(canvas.astype(np.uint8))

_tc3.release()
_writer.release()
shutil.rmtree(_tmp)
print(f'Overlay saved to {overlay_path}')

display(Video(overlay_path, embed=True, width=700))

## 5 — Export Session JSON

Saves all region prompts and the init-frame index to `sam_seg/session.json`.  
This file is consumed by `sam2_propagate.py`.

In [ ]:
active = [rd for rd in REGION_DEFS if regions[rd['name']]['points']]
if not active:
    raise RuntimeError('No prompts found. Complete the prompting cell first.')

session = {
    'speaker':           SPK,
    'initial_frame_idx': ann_frame_idx,
    'regions': [
        {
            'name':   rd['name'],
            'obj_id': rd['obj_id'],
            'color':  rd['color'],
            'points': regions[rd['name']]['points'],
            'labels': regions[rd['name']]['labels'],
        }
        for rd in active
    ],
}

session_path = os.path.join(SAM_SEG_DIR, 'session.json')
with open(session_path, 'w') as f:
    json.dump(session, f, indent=2)

print(f'Session saved to {session_path}')
print()
print(json.dumps(session, indent=2))
print()
print(f'To propagate across all videos run:')
print(f'  python sam2_propagate.py --session {session_path}')